# Customer Feedback Analysis & Automated Apology Emails

## A. Introduction

**Objective:** Build a simple Python solution that (1) finds critical customer reviews without machine learning, (2) shows the most common complaint keywords, and (3) uses Generative AI to draft short, empathetic apology emails for 3 of the most critical reviews.

**Business problem:** A retail company receives thousands of reviews every week. Its support team reads them manually to find the urgent, negative ones and then writes each apology email by hand. This is slow and inconsistent. Automating the filtering and the first draft of the reply lets the team focus on solving problems.

**Dataset:** *Women's E-Commerce Clothing Reviews* (Kaggle). Columns used: `Review Text` (the review) and `Rating` (1-5 stars).

**Run the cells from top to bottom.**

## B. Import Libraries

In [ ]:
import os                                   # read environment variables and check file paths
import re                                   # regular expressions for text cleaning
from collections import Counter             # count keyword frequency

import numpy as np                          # numeric helper library
import pandas as pd                         # data loading and wrangling
import matplotlib.pyplot as plt             # simple bar chart
from IPython.display import display, Markdown  # show formatted output in the notebook

print("Libraries imported successfully.")

## C. Load Dataset

Download the CSV from Kaggle (see the README) and save it as `data/Womens Clothing E-Commerce Reviews.csv`.

In [ ]:
# File path and the two columns we use
DATA_PATH = "data/Womens Clothing E-Commerce Reviews.csv"
REVIEW_COL = "Review Text"
RATING_COL = "Rating"

# Stop with a clear message if the file is missing
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Dataset not found at '{DATA_PATH}'. "
        "Download it from Kaggle and place it in the 'data' folder (see README)."
    )

df = pd.read_csv(DATA_PATH)

# This dataset ships with an extra index column; drop it if present
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

# Make sure the columns we need really exist
for column_name in [REVIEW_COL, RATING_COL]:
    if column_name not in df.columns:
        raise KeyError(f"Column '{column_name}' not found. Available columns: {list(df.columns)}")

print("First 5 rows:")
display(df.head())

print("Shape (rows, columns):", df.shape)
print("Column names:", list(df.columns))

print("\nMissing values per column:")
print(df.isnull().sum())

## D. Data Cleaning

Steps:
1. Keep only the rating and review text columns.
2. Drop rows where the rating or the review text is missing (a review without text cannot be analysed).
3. Clean the text: lowercase, remove special characters, remove extra whitespace.
4. Remove rows that became empty after cleaning.

The original text is kept in `review_original` (used later for the emails); the cleaned text is stored in `review_clean` (used for keyword counting).

In [ ]:
rows_before_cleaning = len(df)

# 1. Keep only the needed columns and give them simple names
df_clean = df[[RATING_COL, REVIEW_COL]].copy()
df_clean = df_clean.rename(columns={RATING_COL: "rating", REVIEW_COL: "review_original"})

# 2. Handle missing values
df_clean["rating"] = pd.to_numeric(df_clean["rating"], errors="coerce")   # non-numeric ratings become NaN
df_clean = df_clean.dropna(subset=["rating", "review_original"])
df_clean = df_clean[df_clean["rating"].between(1, 5)]                     # keep valid 1-5 star ratings only
df_clean["rating"] = df_clean["rating"].astype(int)
df_clean["review_original"] = df_clean["review_original"].astype(str).str.strip()


# 3. Text cleaning function
def clean_text(text):
    text = str(text).lower()                      # lowercase
    text = re.sub(r"[^a-z0-9\s']", " ", text)     # replace special characters with a space (keep apostrophes)
    text = re.sub(r"\s+", " ", text).strip()      # collapse repeated whitespace
    return text


df_clean["review_clean"] = df_clean["review_original"].apply(clean_text)

# 4. Remove reviews that are empty after cleaning
df_clean = df_clean[df_clean["review_clean"] != ""].reset_index(drop=True)

print(f"Rows before cleaning: {rows_before_cleaning}")
print(f"Rows after cleaning:  {len(df_clean)}")
print(f"Rows removed:         {rows_before_cleaning - len(df_clean)}")

print("\nExample (original vs cleaned):")
display(df_clean[["rating", "review_original", "review_clean"]].head(3))

## E. Rule-Based Critical Review Filtering

**Rule:** a review is **Critical** if its rating is **1 or 2 stars**. This is a plain `if`-style rule using pandas, with no machine learning.

In [ ]:
CRITICAL_RATINGS = [1, 2]

# Apply the rule
df_critical = df_clean[df_clean["rating"].isin(CRITICAL_RATINGS)].copy()

total_reviews = len(df_clean)
critical_count = len(df_critical)
critical_percentage = (critical_count / total_reviews * 100) if total_reviews > 0 else 0

print(f"Total reviews (after cleaning): {total_reviews}")
print(f"Critical reviews (1-2 stars):   {critical_count}")
print(f"Percentage of critical reviews: {critical_percentage:.2f}%")

print("\nSample critical reviews:")
display(df_critical[["rating", "review_original"]].head(5))

## F. Complaint Keyword Analysis

Only the critical reviews are analysed. Each cleaned review is split into words, meaningless stopwords (like "the", "is", "and") are removed, and `Counter` counts the rest.

The stopword list is deliberately small and editable. A few generic shopping words (e.g. "ordered", "bought") are also removed because they do not describe a complaint.

In [ ]:
# Common English words that carry no complaint meaning
STOPWORDS = {
    "a", "an", "the", "and", "or", "but", "if", "so", "as", "of", "at", "by", "for", "with", "to", "from",
    "in", "on", "off", "up", "down", "out", "over", "into", "than", "then", "too", "very", "just", "also",
    "is", "am", "are", "was", "were", "be", "been", "being", "do", "does", "did", "have", "has", "had",
    "i", "me", "my", "we", "our", "you", "your", "he", "she", "it", "its", "they", "them", "their",
    "this", "that", "these", "those", "there", "here", "what", "which", "who", "when", "where", "why", "how",
    "not", "no", "yes", "can", "could", "would", "should", "will", "get", "got", "one", "all", "any", "some",
    "more", "most", "much", "many", "even", "only", "about", "after", "before", "because", "while", "still",
    "i'm", "i've", "i'd", "i'll", "it's", "that's", "don't", "didn't", "doesn't", "wasn't", "isn't", "can't",
    "won't", "couldn't", "wouldn't", "they're", "you're", "there's", "let", "am", "such", "really", "like",
}

# Generic shopping words that are not complaints
DOMAIN_STOPWORDS = {"ordered", "bought", "order", "purchase", "purchased", "wear", "wore", "worn", "tried", "buy"}

ALL_STOPWORDS = STOPWORDS | DOMAIN_STOPWORDS


def get_top_keywords(texts, top_n=15, min_word_length=3):
    # Return the top_n most common meaningful words in a list of cleaned texts
    word_counter = Counter()
    for text in texts:
        for word in str(text).split():
            word = word.strip("'")
            if len(word) >= min_word_length and word not in ALL_STOPWORDS and not word.isdigit():
                word_counter[word] += 1
    return word_counter.most_common(top_n)


if critical_count == 0:
    top_keywords = []
    print("No critical reviews found, so there are no keywords to analyse.")
else:
    top_keywords = get_top_keywords(df_critical["review_clean"], top_n=15)

    keywords_table = pd.DataFrame(top_keywords, columns=["keyword", "count"])
    display(keywords_table)

    # Simple horizontal bar chart (reversed so the most common word is on top)
    plt.figure(figsize=(8, 5))
    plt.barh(keywords_table["keyword"][::-1], keywords_table["count"][::-1])
    plt.title("Top Complaint Keywords in Critical Reviews (1-2 stars)")
    plt.xlabel("Number of occurrences")
    plt.tight_layout()
    plt.show()

## G. Select 3 Critical Reviews

**Selection rule (simple and explainable):** from the critical reviews, sort by lowest rating first (1 star = most negative) and then by longest review (more words = more detail about the problem). The top 3 are selected, because detailed complaints give the AI more specific issues to address.

In [ ]:
# Select up to three critical reviews; do not stop the notebook if fewer than three exist.
df_critical["word_count"] = df_critical["review_clean"].str.split().str.len()
selected_reviews = (
    df_critical.sort_values(by=["rating", "word_count"], ascending=[True, False])
    .head(3)
    .reset_index(drop=True)
)

if selected_reviews.empty:
    print("No critical reviews are available, so no apology emails will be generated.")
else:
    print(f"Selected {len(selected_reviews)} critical review(s) for draft responses.")
    for position, row in selected_reviews.iterrows():
        print(f"--- Selected review {position + 1} | rating: {row['rating']} star(s) | words: {row['word_count']} ---")
        print(row["review_original"])
        print()


## H. Generative AI - Apology Emails

**API key safety:** the key is read from the `OPENAI_API_KEY` environment variable and is never typed into the notebook or printed. When enabled, review text is sent to an external API provider. Only send data you are authorized to share; do not transmit confidential or identifiable customer information.

**If no key is available** (or the API call fails), the notebook falls back to a simple template. Template emails are clearly labelled **MOCK - NOT AI GENERATED** and are only there to show the expected format.

In [ ]:
MODEL_NAME = "gpt-4o-mini"   # change to any chat model available on your account

api_key = os.getenv("OPENAI_API_KEY")
client = None
USE_REAL_API = False

if not api_key:
    print("OPENAI_API_KEY not found. The MOCK fallback will be used (emails will NOT be AI generated).")
else:
    try:
        from openai import OpenAI
        client = OpenAI()        # the SDK reads OPENAI_API_KEY from the environment automatically
        USE_REAL_API = True
        print(f"OpenAI client ready (model: {MODEL_NAME}). Real API calls will be attempted.")
    except ImportError:
        print("The 'openai' package is not installed (run: pip install openai). Using MOCK fallback.")

In [ ]:
SYSTEM_PROMPT = (
    "You are a friendly, professional Customer Support Agent for a retail company. "
    "Write a SHORT apology email (under 150 words) in response to the customer review you are given. "
    "Rules: "
    "1) Start with a subject line. "
    "2) Be warm, empathetic and personal. "
    "3) Refer specifically to the problems mentioned in the review. "
    "4) Do NOT promise refunds, replacements, discounts or compensation, and do not invent order details or policies. "
    "5) Invite the customer to reply so the team can look into the issue. "
    "6) Sign off as 'Customer Support Team'."
)


def generate_email_with_api(review_text):
    # Send one review to the OpenAI API and return the generated email text
    response = client.responses.create(
        model=MODEL_NAME,
        instructions=SYSTEM_PROMPT,
        input=f"Customer review:\n{review_text}\n\nWrite the apology email now.",
    )
    return response.output_text.strip()


def generate_mock_email(review_text):
    # Template fallback. NOT produced by an AI model.
    keywords = [word for word, _ in get_top_keywords([clean_text(review_text)], top_n=3)]
    issues = ", ".join(keywords) if keywords else "the problems you described"
    return (
        "[MOCK EMAIL - NOT AI GENERATED]\n"
        "Subject: We are sorry about your experience\n\n"
        "Dear Customer,\n\n"
        "Thank you for your feedback, and we are sorry your experience did not meet your expectations. "
        f"We noted the concerns you raised (key words in your review: {issues}). "
        "Please reply to this email with any extra details so our team can look into it.\n\n"
        "Kind regards,\nCustomer Support Team"
    )


generated_emails = []

for position, row in selected_reviews.iterrows():
    review_text = row["review_original"]
    email_text = None
    email_source = None

    if USE_REAL_API:
        try:
            email_text = generate_email_with_api(review_text)
            email_source = f"OpenAI API ({MODEL_NAME})"
        except Exception as error:     # e.g. invalid key, no quota, network problem
            print(f"API call failed for review {position + 1}: {type(error).__name__}: {error}")

    if email_text is None:             # fallback when there is no key or the call failed
        email_text = generate_mock_email(review_text)
        email_source = "MOCK template (not AI generated)"

    generated_emails.append({
        "review_number": position + 1,
        "rating": row["rating"],
        "review": review_text,
        "email": email_text,
        "source": email_source,
    })
    print(f"Review {position + 1} done. Source: {email_source}")

## I. Results

The cell below builds the results as formatted Markdown from the variables calculated above, so the numbers always match the real data.

In [ ]:
def quote(text):
    # Format multi-line text as a Markdown blockquote
    return "> " + str(text).replace("\n", "\n> ")


# Dataset statistics
rating_counts = df_clean["rating"].value_counts().sort_index()
rating_lines = "\n".join(f"- {stars} star(s): {count}" for stars, count in rating_counts.items())

results_md = f"""
### Dataset statistics
- Raw rows loaded: **{rows_before_cleaning}**
- Rows after cleaning: **{total_reviews}**
- Average rating: **{df_clean['rating'].mean():.2f}**

Rating distribution:
{rating_lines}

### Critical review summary (1-2 stars)
- Number of critical reviews: **{critical_count}**
- Percentage of critical reviews: **{critical_percentage:.2f}%**

### Top complaint keywords
""" + "\n".join(f"{i}. **{word}** - {count}" for i, (word, count) in enumerate(top_keywords, start=1))

display(Markdown(results_md))

display(Markdown("### The 3 selected critical reviews"))
for item in generated_emails:
    display(Markdown(f"**Review {item['review_number']}** ({item['rating']} star(s))\n\n{quote(item['review'])}"))

display(Markdown("### The 3 customer-support emails"))
for item in generated_emails:
    display(Markdown(f"**Email {item['review_number']}** - source: *{item['source']}*\n\n{quote(item['email'])}"))

## J. Conclusion

This notebook loaded and cleaned a customer-review dataset, isolated critical reviews (1-2 stars) with a simple rule, counted the most common complaint keywords, and produced personalized apology emails for 3 detailed complaints. Check the source label under each email: it shows whether it came from the OpenAI API or from the mock template.

**How automation helps a support team:** the rule-based filter removes the need to read every review, keyword counts show recurring problems (for example product quality or sizing) at a glance, and AI-drafted emails give agents a fast, consistent first draft. Agents should still review each draft before sending, and add refund or replacement details only where company policy allows.